# ML 기반 다변량 이상 탐지 (Multivariate Anomaly Detection)

In [182]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn import set_config
set_config(display='text')

## 가상 Dry Etch 정상/비정상 데이터 생성
- 정상 : 2000
- 비정상 : 50

In [183]:
np.random.seed(77)
n_normal = 2000
n_anomaly = 50

### 정상 데이터 설정
- 상황 가정 : RF Power & Temperature 가 관련있는 변수
    - RF Power 증가 -> 플라즈마 생성, 열 발생 -> 온도 증가
    - 선형적인 인과관계 가정 (RF Power 증가시 온도 증가하는 과정 반영)

In [184]:
normal_rf = np.random.normal(600, 10, n_normal)
normal_temp = normal_rf * 0.15 + np.random.normal(80, 1, n_normal)
X_normal = pd.DataFrame({"RF_Power": normal_rf, "Temp": normal_temp})

### 비정상 데이터 설정
- 상황 가정 : 쿨러에서 사용되는 Fluid 누수
    - 적절히 Cooling이 되지 않아 RF Power 에 비해 Temp가 기존 관계보다 과열 되는 상황

In [185]:
anomaly_rf = np.random.normal(610, 5, n_anomaly)
anomaly_temp = np.random.normal(185, 5, n_anomaly)
X_anomaly = pd.DataFrame({"RF_Power": anomaly_rf, "Temp": anomaly_temp})
X_all = pd.concat([X_normal, X_anomaly], ignore_index=True)

## 데이터 정규화
- ML 학습 전 데이터에 대한 전처리
    - 각 데이터에 대한 스케일 차이 제거
    - 학습 안정성, 수렴 속도

In [186]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_all)

## Autoencoder 모델 도입
- MLP Regressor (Multi-Layer Perceptron) 기반으로 모델 구현
    - 입력층 -> 은닉층 -> 출력층 구조
    - 은닉층 1개 -> tanh (비선형 함수 사용)
    - autoencoder 구현 : 입력, 출력을 각각 X_train으로 같은 값 사용, 복원 여부를 판단

In [187]:
X_train = X_scaled[:1800] # 정상 데이터 일부로 학습
autoencoder = MLPRegressor(
    hidden_layer_sizes=(2,), # 병목층(Latent Layer)
    activation="tanh",
    solver="adam",
    max_iter=300,
    random_state=42
)
autoencoder.fit(X_train, X_train)

MLPRegressor(activation='tanh', hidden_layer_sizes=(2,), max_iter=300,
             random_state=42)

## 복원 오차 계산

In [188]:
X_reconstructed = autoencoder.predict(X_scaled)
reconstruction_error = np.mean(np.square(X_scaled - X_reconstructed), axis=1)
X_all["Recon_Loss"] = reconstruction_error

## 임계치 적용

In [189]:
threshold = np.percentile(reconstruction_error[:n_normal], 99.5)
X_all["Anomaly_Detected"] = X_all["Recon_Loss"] > threshold

print(f"Reconstruction Loss Threshold: {threshold:.5f}")
print(f"Detected Abnormal Run Count: {X_all['Anomaly_Detected'].sum()} / 2050")

Reconstruction Loss Threshold: 0.01358
Detected Abnormal Run Count: 54 / 2050


In [190]:
# 정상 데이터 중 오탐된 인덱스
false_positive_idx = X_all[(X_all.index < 2000) & (X_all["Anomaly_Detected"])].index.tolist()

# 실제 이상치 중 탐지된 인덱스
true_positive_idx = X_all[(X_all.index >= 2000) & (X_all["Anomaly_Detected"])].index.tolist()

# 실제 이상치인데 놓친 인덱스
false_negative_idx = X_all[(X_all.index >= 2000) & (~X_all["Anomaly_Detected"])].index.tolist()

print("오탐(정상인데 이상으로 잡힘):", false_positive_idx)
print("정탐(실제 이상치 잡음):", true_positive_idx)
print("미탐(실제 이상치인데 놓침):", false_negative_idx)

오탐(정상인데 이상으로 잡힘): [64, 166, 440, 568, 591, 945, 1073, 1407, 1908, 1976]
정탐(실제 이상치 잡음): [2000, 2001, 2002, 2003, 2004, 2006, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2038, 2039, 2040, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049]
미탐(실제 이상치인데 놓침): [2005, 2007, 2008, 2029, 2037, 2041]
